# PA3 Task 1 on Kaggle — benchmark evaluation

Runs `task1.evaluate` for the conditions below, **one per T4 in parallel** (GSM8K, then HellaSwag/BoolQ/ARC-Easy),
with the same settings as the local runs (generation batch 32, log-likelihood batch 4, same library versions).

**Before running:**
1. Right-hand panel → *Session options* → **Accelerator: GPU T4 x2**, **Internet: On**.
2. Right-hand panel → **Add Input** for each condition's trained weights:
   - `full_ft_l2sp` → *Your Work* → *Notebooks* → the notebook that trained `full_ft_l2sp` (its output has `outputs/task1/runs/full_ft_l2sp/exports/u1250/`).
   - `oplora16` → *Your Work* → *Datasets* → the dataset you uploaded from `kaggle_upload/oplora16.zip`.
3. **Save Version → Save & Run All (Commit)**. About 3–4 h; you can close the tab.

Then open the saved version → **Output** → download `eval_results.zip` and unzip it into `outputs/task1/eval/` locally.

In [ ]:
CONDITIONS = ["full_ft_l2sp", "oplora16"]   # condition i runs on GPU i % 2; more than two run one after another per GPU

In [ ]:
!git clone -q https://github.com/ahmadhassann96/ATML_PA3.git /kaggle/working/ATML_PA3
%cd /kaggle/working/ATML_PA3
!git log --oneline -1
# same versions as the local evaluations
!pip install -q "transformers==4.57.1" "lm_eval[hf]==0.4.13"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
# Find each condition's exported weights among the attached inputs and link them where task1 expects them.
import glob, os, zipfile
from pathlib import Path

os.environ["PA3_OUT"] = "/tmp/pa3_out"          # results go to /tmp; only the final zip is saved as output
SPEC = {"full_ft@625": ("full_ft", 625), "full_ft": ("full_ft", 1250), "full_ft_l2sp": ("full_ft_l2sp", 1250),
        "lora4": ("lora4", 1250), "lora16": ("lora16", 1250), "oplora16": ("oplora16", 1250)}

def find_export(method, update):
    pat = f"**/{method}/exports/u{update}"
    hits = glob.glob(f"/kaggle/input/{pat}", recursive=True)
    if not hits:   # inputs that are still zipped (e.g. the training notebook's <method>.zip)
        for z in glob.glob(f"/kaggle/input/**/{method}.zip", recursive=True):
            zipfile.ZipFile(z).extractall(f"/tmp/unzipped/{method}")
        hits = glob.glob(f"/tmp/unzipped/{method}/{pat}", recursive=True)
    return hits[0] if hits else None

for c in CONDITIONS:
    method, update = SPEC[c]
    src = find_export(method, update)
    assert src, f"{c}: no {method}/exports/u{update} in /kaggle/input -- add the right input (see the top cell)"
    dst = Path(f"/tmp/pa3_out/task1/runs/{method}/exports/u{update}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        os.symlink(src, dst)
    print(f"{c}: {src} -> {sorted(os.listdir(src))}")

In [ ]:
# Launch one worker per GPU; each evaluates its conditions in order. Logs: /kaggle/working/eval_<condition>.log
import subprocess, sys, time

workers = []
for g in range(2):
    mine = CONDITIONS[g::2]
    if not mine:
        continue
    for c in mine:
        log = f"/kaggle/working/eval_{c.replace('@', '_at_')}.log"
        cmd = [sys.executable, "-m", "task1.evaluate", "--condition", c]
        workers.append((g, c, log, cmd))

def run_gpu(g):
    for gg, c, log, cmd in workers:
        if gg == g:
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
            with open(log, "w") as f:
                rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=env).returncode
            print(f"[GPU {g}] {c} finished with exit code {rc}", flush=True)

import threading
threads = [threading.Thread(target=run_gpu, args=(g,)) for g in sorted({w[0] for w in workers})]
for t in threads:
    t.start()
while any(t.is_alive() for t in threads):
    time.sleep(600)
    for g, c, log, _ in workers:
        if os.path.exists(log):
            tail = open(log, errors="ignore").read()[-400:].replace("\r", "\n").strip().splitlines()
            print(time.strftime("%H:%M"), f"GPU {g} {c}:", tail[-1][:150] if tail else "", flush=True)
for t in threads:
    t.join()

In [ ]:
# Summary + one zip to download (unzip into outputs/task1/eval/ locally)
import json
for c in CONDITIONS:
    p = Path(f"/tmp/pa3_out/task1/eval/{c.replace('@', '_at_')}/summary.json")
    s = json.loads(p.read_text()) if p.exists() else {}
    print(c, "complete" if "primary" in s else "INCOMPLETE -- check its log", s.get("primary"), s.get("run_info"))
!cd /tmp/pa3_out/task1/eval && zip -r -q /kaggle/working/eval_results.zip .
!ls -lh /kaggle/working/eval_results.zip